# 04 RAG Agent End-to-End

Run a full agentic RAG workflow from one annual report: extract markdown with Azure Document Intelligence, build table-aware chunks, optionally persist chunks to Postgres/pgvector, inspect retrieval results, and invoke the LangChain RAG agent for a fundamentals question.

## Setup

This notebook stays thin. Document extraction, chunking, vector persistence, retrieval, and agent construction all come from reusable project modules.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import Markdown, display

from backend import ingest_reports
from market_analyst.config.settings import load_settings
from market_analyst.repositories.vector_db import full_text_search, hybrid_search, vector_search
from market_analyst.services.agent import build_market_analysis_agent, format_retrieval_results
from market_analyst.services.rag import discover_reports
from market_analyst.telemetry import configure_notebook_logging

settings = load_settings()
logger = configure_notebook_logging(run_name="04_rag_agent_end_to_end")
print("Vector collection:", settings.vector_collection_name)

c:\Users\rushi\OneDrive - ImmersiLearn Education Services LLP\Projects\LLM Projects\market-analyst-feb26\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
2026-05-16 10:54:10,795 INFO 04_rag_agent_end_to_end notebook_run_started


Vector collection: fundamental_report_chunks


## Run Configuration

Set the document and retrieval options here. `PERSIST_TO_VECTOR_DB = True` writes chunks to both the LangChain PGVector collection and the project `reports` table used for full-text search.

In [2]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORT_INDEX = 0
MAX_PAGES = None
CHUNK_SIZE = 1400
CHUNK_OVERLAP = 180
PERSIST_TO_VECTOR_DB = True
RESET_VECTOR_COLLECTION = False
RETRIEVAL_LIMIT = 5

QUESTION = "What do the annual report chunks say about growth, debt, cash flow, and key business risks?"

## Choose A Document

The notebook discovers PDF files from `reports/`. Put the annual report PDF there, then choose `REPORT_INDEX` above.

In [3]:
reports = discover_reports(REPORTS_DIR)
assert reports, f"No PDF reports found in {REPORTS_DIR}"

reports_df = pd.DataFrame(
    [
        {
            "index": index,
            "ticker": report.ticker,
            "company_name": report.company_name,
            "file": report.path.name,
            "path": str(report.path),
        }
        for index, report in enumerate(reports)
    ]
)
display(reports_df)

selected_report = reports[REPORT_INDEX]
print("Selected:", selected_report.ticker, selected_report.company_name, selected_report.path.name)

,index,ticker,company_name,file,path
0,0,BANDHAN,Bandhan,bandhan_annual_report.pdf,c:\Users\rushi\OneDrive - ImmersiLearn Educati...
1,1,BANDHANQUARTERLY,Bandhan Quarterly,bandhan_quarterly_report.pdf,c:\Users\rushi\OneDrive - ImmersiLearn Educati...
2,2,EMCURE,Emcure,emcure_annual_report.pdf,c:\Users\rushi\OneDrive - ImmersiLearn Educati...


Selected: BANDHAN Bandhan bandhan_annual_report.pdf


## Extract, Chunk, And Persist

This cell calls the shared backend ingestion path. It uses Azure Document Intelligence for markdown extraction, the service-layer table-aware RAG splitter for chunks, and the repository layer for vector/full-text persistence.

In [4]:
result = ingest_reports(
    reports=[selected_report],
    max_pages=MAX_PAGES,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    persist=PERSIST_TO_VECTOR_DB,
    reset_collection=RESET_VECTOR_COLLECTION,
)

print(f"reports: {result.report_count}")
print(f"chunks: {result.chunk_count}")
print(f"vector ids: {len(result.vector_ids)}")
print(f"project report rows: {result.reports_rows}")

2026-05-10 12:02:04,722 INFO azure.core.pipeline.policies.http_logging_policy Request URL: 'https://vector-database.cognitiveservices.azure.com//documentintelligence/documentModels/prebuilt-layout:analyze?api-version=2024-11-30&outputContentFormat=REDACTED'
Request method: 'POST'
Request headers:
    'content-type': 'application/json'
    'Content-Length': '2023448'
    'Accept': 'application/json'
    'x-ms-client-request-id': 'f5e1374b-4c39-11f1-a24e-24b2b95b580e'
    'x-ms-useragent': 'REDACTED'
    'User-Agent': 'azsdk-python-ai-documentintelligence/1.0.2 Python/3.13.5 (Windows-11-10.0.26200-SP0)'
    'Ocp-Apim-Subscription-Key': 'REDACTED'
A body is sent with the request
2026-05-10 12:02:08,038 INFO azure.core.pipeline.policies.http_logging_policy Response status: 202
Response headers:
    'Content-Length': '0'
    'Operation-Location': 'REDACTED'
    'x-envoy-upstream-service-time': 'REDACTED'
    'apim-request-id': 'REDACTED'
    'Strict-Transport-Security': 'REDACTED'
    'x-co

reports: 1
chunks: 1847
vector ids: 1847
project report rows: 1847


## Markdown Preview

Inspect the extracted markdown before retrieval. This is useful for checking whether tables and section headings came through cleanly.

In [5]:
markdown_report = result.markdown_reports[0]
print(markdown_report.report.path.name, "pages:", markdown_report.page_count)
display(Markdown(markdown_report.markdown[:6000]))

bandhan_annual_report.pdf pages: 300


# Bandhan
## bandhan_annual_report.pdf

### Page 0
<figure>

Bandhan
Bank

</figure>


#### Trust that Grows, Bandhan that Strengthens

<!-- PageFooter="Annual Report 2024-25" -->
<!-- PageBreak -->

### Page 0
<!-- PageBreak -->

### Page 0
#### Contents


<table>
<tr>
<th colspan="2">Corporate Overview</th>
</tr>
<tr>
<td>About the Report</td>
<td>02</td>
</tr>
<tr>
<td>Trust that Grows,</td>
<td></td>
</tr>
<tr>
<td>Bandhan that Strengthens</td>
<td>04</td>
</tr>
<tr>
<td>Company Overview</td>
<td></td>
</tr>
<tr>
<td>A Growing Network, A Deeper Commitment</td>
<td>06</td>
</tr>
<tr>
<td>Nationwide Footprint</td>
<td></td>
</tr>
<tr>
<td>Strengthening Trust Across India</td>
<td>08</td>
</tr>
<tr>
<td>Milestone</td>
<td></td>
</tr>
<tr>
<td>Our Journey Over the Years</td>
<td>10</td>
</tr>
<tr>
<td>FY25 Highlights</td>
<td></td>
</tr>
<tr>
<td>Trust Reflected in Numbers</td>
<td>12</td>
</tr>
<tr>
<td>Key Performance Indicators</td>
<td></td>
</tr>
<tr>
<td>Sustaining the Bandhan in Numbers</td>
<td>14</td>
</tr>
<tr>
<td>Management Messages</td>
<td></td>
</tr>
<tr>
<td>Chairman's Message</td>
<td>18</td>
</tr>
<tr>
<td>MD and CEO's Message</td>
<td>20</td>
</tr>
<tr>
<td>Governance</td>
<td></td>
</tr>
<tr>
<td>Board of Directors</td>
<td>24</td>
</tr>
<tr>
<td>Senior Management Team</td>
<td>31</td>
</tr>
<tr>
<td>Leadership Team</td>
<td>31</td>
</tr>
<tr>
<td>Diversified Product Suite</td>
<td></td>
</tr>
<tr>
<td rowspan="2">Tailored Offerings: Built on Bandhan, Designed for Progress</td>
<td></td>
</tr>
<tr>
<td>32</td>
</tr>
<tr>
<td>Information Technology</td>
<td></td>
</tr>
<tr>
<td>Fortifying Digital Trust through Scalable Tech</td>
<td>48</td>
</tr>
<tr>
<td>Operational Excellence</td>
<td></td>
</tr>
<tr>
<td>Cementing Trust Through</td>
<td></td>
</tr>
<tr>
<td>Operational Strength</td>
<td>50</td>
</tr>
<tr>
<td>Customer Experience</td>
<td></td>
</tr>
<tr>
<td rowspan="2">Quality in Every Interaction and Delivering Best-in-Class Banking Experience</td>
<td></td>
</tr>
<tr>
<td>52</td>
</tr>
</table>


Transformation Management Office
The Epicentre of Trust-led Transformation
54

Human Capital

Embedding a Culture of Care,
Capability, and Collaboration
56

Risk Management

Focusing on what Matters:
Key Risk Areas
62

Credit Underwriting

Building a Disciplined and Feasible

Credit Underwriting Framework

69

Compliance

Compliance First:
Ensuring Integrity, Earning Trust
70

Data Analytics

Data-Driven Approach:

Fostering Trust Through Analytics

72

Marketing

Championing Bandhan: Marketing

that Connects, Messaging that Endures

74

Corporate Social Responsibility

Driven by Values, Defined by Impact

78

Stories of Transformation

Bandhan in Action:

Lending Vision, Transforming Lives

85

Statutory Reports


<table>
<tr>
<th>Board's Report</th>
<th>88</th>
</tr>
<tr>
<td>Report on Corporate Governance</td>
<td>125</td>
</tr>
<tr>
<td>Management Discussion &amp;</td>
<td></td>
</tr>
<tr>
<td>Analysis Report</td>
<td>170</td>
</tr>
<tr>
<td rowspan="2">Business Responsibility &amp; Sustainability Report</td>
<td></td>
</tr>
<tr>
<td>176</td>
</tr>
</table>


Financial Statements

<!-- PageNumber="224" -->
<!-- PageBreak -->

### Page 0
<!-- PageHeader="Annual Report 2024-25" -->


<figure>

Bandhan
Bank

</figure>


#### About the Report


#### Reporting Objective

As an entity deeply committed to upholding a legacy of
trust, integrity, and excellence, your Bank's journey in
FY25 reflects its unwavering focus on customer centricity,
ethical governance, and long-term value creation. Your
Bank's key purpose is to ensure that every individual falls
under the purview of formal banking. To achieve this,
your Bank is providing universal banking services to the
unbanked and underbanked, while also serving metro and
urban customers through its products and services. This
reflects your Bank's commitment to its purpose.

The objective of this Annual Report is to communicate
your Bank's FY25 financial performance, strategic
priorities, and operational successes openly and
comprehensively. Through this Report, your Bank seeks
to strengthen stakeholder trust, reaffirm its commitment
to ethical governance, and showcase its brand values
translate into meaningful impact for its customers,
partners, and the wider community.


#### Scope and Boundary

This Annual Report presents a comprehensive
account of the financial and non-financial
performance of your Bank for the period of
April 1, 2024 to March 31, 2025.

It offers a holistic view of your Bank's operations,
outlining the key activities that drive short-
term and long-term value creation for its
stakeholders. The Report also covers your Bank's
product portfolio, competitive positioning,
strategic priorities, business model, and risk
management framework.

<!-- PageNumber="02" -->
<!-- PageBreak -->

### Page 0
<figure>

<!-- PageHeader="Corporate Overview" -->

</figure>


<!-- PageHeader="Statutory Reports" -->
<!-- PageHeader="Financial Statements" -->


#### Reporting Framework

This Report has been prepared as per the
following guiding principles, encompassing
strategic focus, future orientation, information
connectivity, stakeholder engagement, and
consistency to ensure transparency and long-term
value communication.


#### Board Approval

The Board of Directors affirms that it has collectively
applied its mind to the preparation and presentation
of this Annual Report. The Board accepts responsibility
for the integrity and completeness of the Report.
It is of the view that it fairly, transparently, and
accurately reflects all material matters and your Bank's
financial and non-financial performance during the
reporting period. This Report has been prepared in
accordance with applicable laws, regulations, and
reporting frameworks.


#### Forward-looking Statement

Certain statements in the Report regarding your Bank's business operations may constitute
forward-looking statements. They may be identified by their use of words 

## Chunk Inventory

The chunk table shows section metadata, table preservation flags, and a short preview for each chunk.

In [6]:
chunk_rows = [
    {
        "chunk_id": chunk.id,
        "ticker": chunk.metadata.get("ticker"),
        "page": chunk.metadata.get("page_number"),
        "heading_path": chunk.metadata.get("heading_path"),
        "contains_table": bool(chunk.metadata.get("contains_table")),
        "table_format": chunk.metadata.get("table_format"),
        "chars": len(chunk.page_content),
        "preview": chunk.page_content[:240].replace("\n", " "),
    }
    for chunk in result.chunks
]
chunks_df = pd.DataFrame(chunk_rows)
display(chunks_df)

,chunk_id,ticker,page,heading_path,contains_table,table_format,chars,preview
0,bandhan-00000-1bab5500ed103fbf,BANDHAN,0,Bandhan > bandhan_annual_report.pdf > Page 0,False,NaN,89,# Bandhan ## bandhan_annual_report.pdf ###...
1,bandhan-00001-90065d59470a7cb6,BANDHAN,0,Bandhan > bandhan_annual_report.pdf > Page 0 >...,False,NaN,112,"#### Trust that Grows, Bandhan that Strengthen..."
2,bandhan-00002-5d6b159fbe9a24b9,BANDHAN,0,Bandhan > bandhan_annual_report.pdf > Page 0 >...,False,NaN,58,### Page 0 <!-- PageBreak --> ### Page 0 #...
3,bandhan-00003-4680edd4d982daa1,BANDHAN,0,Bandhan > bandhan_annual_report.pdf > Page 0 >...,True,html,2191,### Page 0 <!-- PageBreak --> ### Page 0 #### ...
4,bandhan-00004-cf6b0018efddf412,BANDHAN,0,Bandhan > bandhan_annual_report.pdf > Page 0 >...,False,NaN,748,Transformation Management Office The Epicentre...
...,...,...,...,...,...,...,...,...
1842,bandhan-01842-095757e10402333a,BANDHAN,290,Bandhan > bandhan_annual_report.pdf > Page 290,False,NaN,94,"### Page 290 <!-- PageHeader=""Annual Report 20..."
1843,bandhan-01843-129655c20805df9a,BANDHAN,290,Bandhan > bandhan_annual_report.pdf > Page 290...,False,NaN,26,#### Corporate Information
1844,bandhan-01844-80b78518a0be9255,BANDHAN,290,Bandhan > bandhan_annual_report.pdf > Page 290...,True,html,2120,#### Corporate Information <table> <tr> <th>R...
1845,bandhan-01845-e953892f2ce79ae3,BANDHAN,290,Bandhan > bandhan_annual_report.pdf > Page 290...,False,NaN,44,"<!-- PageNumber=""296"" --> <!-- PageBreak -->"


## Table Chunk Inspection

Table-aware chunks should preserve the full table plus nearby before/after context.

In [7]:
table_chunks = [chunk for chunk in result.chunks if chunk.metadata.get("contains_table")]
print("table chunks:", len(table_chunks))

if table_chunks:
    table_chunk = table_chunks[0]
    display(Markdown(f"### {table_chunk.metadata.get('heading_path')}\n\n{table_chunk.page_content[:5000]}"))
else:
    print("No table chunks detected in the selected page range.")


table chunks: 274


### Bandhan > bandhan_annual_report.pdf > Page 0 > Contents

### Page 0 <!-- PageBreak --> ### Page 0 #### Contents

<table>
<tr>
<th colspan="2">Corporate Overview</th>
</tr>
<tr>
<td>About the Report</td>
<td>02</td>
</tr>
<tr>
<td>Trust that Grows,</td>
<td></td>
</tr>
<tr>
<td>Bandhan that Strengthens</td>
<td>04</td>
</tr>
<tr>
<td>Company Overview</td>
<td></td>
</tr>
<tr>
<td>A Growing Network, A Deeper Commitment</td>
<td>06</td>
</tr>
<tr>
<td>Nationwide Footprint</td>
<td></td>
</tr>
<tr>
<td>Strengthening Trust Across India</td>
<td>08</td>
</tr>
<tr>
<td>Milestone</td>
<td></td>
</tr>
<tr>
<td>Our Journey Over the Years</td>
<td>10</td>
</tr>
<tr>
<td>FY25 Highlights</td>
<td></td>
</tr>
<tr>
<td>Trust Reflected in Numbers</td>
<td>12</td>
</tr>
<tr>
<td>Key Performance Indicators</td>
<td></td>
</tr>
<tr>
<td>Sustaining the Bandhan in Numbers</td>
<td>14</td>
</tr>
<tr>
<td>Management Messages</td>
<td></td>
</tr>
<tr>
<td>Chairman's Message</td>
<td>18</td>
</tr>
<tr>
<td>MD and CEO's Message</td>
<td>20</td>
</tr>
<tr>
<td>Governance</td>
<td></td>
</tr>
<tr>
<td>Board of Directors</td>
<td>24</td>
</tr>
<tr>
<td>Senior Management Team</td>
<td>31</td>
</tr>
<tr>
<td>Leadership Team</td>
<td>31</td>
</tr>
<tr>
<td>Diversified Product Suite</td>
<td></td>
</tr>
<tr>
<td rowspan="2">Tailored Offerings: Built on Bandhan, Designed for Progress</td>
<td></td>
</tr>
<tr>
<td>32</td>
</tr>
<tr>
<td>Information Technology</td>
<td></td>
</tr>
<tr>
<td>Fortifying Digital Trust through Scalable Tech</td>
<td>48</td>
</tr>
<tr>
<td>Operational Excellence</td>
<td></td>
</tr>
<tr>
<td>Cementing Trust Through</td>
<td></td>
</tr>
<tr>
<td>Operational Strength</td>
<td>50</td>
</tr>
<tr>
<td>Customer Experience</td>
<td></td>
</tr>
<tr>
<td rowspan="2">Quality in Every Interaction and Delivering Best-in-Class Banking Experience</td>
<td></td>
</tr>
<tr>
<td>52</td>
</tr>
</table>

Transformation Management Office The Epicentre of Trust-led Transformation 54 Human Capital Embedding a Culture of Care, Capability, and Collaboration 56 Risk Management Focusing on what Matters: Key Risk Areas 62 Credit Underwriting Building a Disciplined and Feasible Credit Underwriting Framework 69 Compliance Compliance First: Ensur

## Retrieval Smoke Test

Run full-text, vector, and fused hybrid retrieval separately. This makes it clear what evidence the RAG agent can use.

In [8]:
QUERY = QUESTION
TICKER_FILTER = selected_report.ticker

assert PERSIST_TO_VECTOR_DB, "Set PERSIST_TO_VECTOR_DB = True and rerun ingestion before retrieval."

full_text_results = full_text_search(settings, QUERY, ticker=TICKER_FILTER, limit=RETRIEVAL_LIMIT)
vector_results = vector_search(settings, QUERY, ticker=TICKER_FILTER, limit=RETRIEVAL_LIMIT)
hybrid_results = hybrid_search(settings, QUERY, ticker=TICKER_FILTER, limit=RETRIEVAL_LIMIT)

def results_frame(rows):
    return pd.DataFrame(
        [
            {
                "ticker": row.get("ticker"),
                "company_name": row.get("company_name"),
                "heading_path": (row.get("metadata") or {}).get("heading_path"),
                "full_text_rank": row.get("full_text_rank"),
                "vector_distance": row.get("vector_distance"),
                "rrf_score": row.get("rrf_score"),
                "content": str(row.get("content", ""))[:300].replace("\n", " "),
            }
            for row in rows
        ]
    )

print("Full-text results")
display(results_frame(full_text_results))
print("Vector results")
display(results_frame(vector_results))
print("Hybrid RRF results")
display(results_frame(hybrid_results))

2026-05-10 12:27:33,065 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-06-01 "HTTP/1.1 200 OK"
2026-05-10 12:27:44,535 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-06-01 "HTTP/1.1 200 OK"


Full-text results


""


Vector results


,ticker,company_name,heading_path,full_text_rank,vector_distance,rrf_score,content
0,BANDHAN,Bandhan,Bandhan > bandhan_annual_report.pdf > Page 0 >...,None,0.496933,None,#### Forward-looking Statement Certain state...
1,BANDHAN,Bandhan,Bandhan > bandhan_annual_report.pdf > Page 170...,None,0.507927,None,#### Management Discussion & Analysis Report
2,BANDHAN,Bandhan,Bandhan > bandhan_annual_report.pdf > Page 0 >...,None,0.512642,None,#### Scope and Boundary This Annual Report p...
3,BANDHAN,Bandhan,Bandhan > bandhan_annual_report.pdf > Page 180...,None,0.518380,None,relevant risks and opportunities that are ment...
4,BANDHAN,Bandhan,Bandhan > bandhan_annual_report.pdf > Page 20,None,0.525533,None,"### Page 20 <figure> <!-- PageHeader=""Corpor..."


Hybrid RRF results


,ticker,company_name,heading_path,full_text_rank,vector_distance,rrf_score,content
0,BANDHAN,Bandhan,Bandhan > bandhan_annual_report.pdf > Page 0 >...,None,0.496933,0.016393,#### Forward-looking Statement Certain state...
1,BANDHAN,Bandhan,Bandhan > bandhan_annual_report.pdf > Page 170...,None,0.507927,0.016129,#### Management Discussion & Analysis Report
2,BANDHAN,Bandhan,Bandhan > bandhan_annual_report.pdf > Page 0 >...,None,0.512642,0.015873,#### Scope and Boundary This Annual Report p...
3,BANDHAN,Bandhan,Bandhan > bandhan_annual_report.pdf > Page 180...,None,0.518380,0.015625,relevant risks and opportunities that are ment...
4,BANDHAN,Bandhan,Bandhan > bandhan_annual_report.pdf > Page 20,None,0.525533,0.015385,"### Page 20 <figure> <!-- PageHeader=""Corpor..."


## Retrieved Context Preview

This is the formatted context string the agent tool returns to the model.

In [9]:
retrieval_context = format_retrieval_results(hybrid_results)
display(Markdown("```text\n" + retrieval_context[:5000] + "\n```"))


```text
Result 1
Ticker: BANDHAN
Company: Bandhan
Section: Bandhan > bandhan_annual_report.pdf > Page 0 > Forward-looking Statement
RRF Score: 0.01639344262295082
Full-text Rank: None
Vector Distance: 0.4969332262089865
Content: #### Forward-looking Statement Certain statements in the Report regarding your Bank's business operations may constitute forward-looking statements. They may be identified by their use of words like 'plans', 'expects', 'will', 'anticipates', 'believes', 'intends', 'projects', 'estimates', or other words of similar meaning. All statements that address expectations or projections about the future, including but not limited to statements about your Bank's strategy for growth, product development, market position, expenditures and financial results, are forward-looking statements. While these statements reflect your Bank's future expectations, it is important to be mindful that some of the risks, uncertainties and other important factors can cause actual results to differ materially from the expectations. Your Bank does not assume responsibility to publicly amend, modify, or revise any forwa...

Result 2
Ticker: BANDHAN
Company: Bandhan
Section: Bandhan > bandhan_annual_report.pdf > Page 170 > Management Discussion & Analysis Report
RRF Score: 0.016129032258064516
Full-text Rank: None
Vector Distance: 0.5079269900465913
Content: #### Management Discussion & Analysis Report

Result 3
Ticker: BANDHAN
Company: Bandhan
Section: Bandhan > bandhan_annual_report.pdf > Page 0 > Scope and Boundary
RRF Score: 0.015873015873015872
Full-text Rank: None
Vector Distance: 0.5126415387569618
Content: #### Scope and Boundary This Annual Report presents a comprehensive account of the financial and non-financial performance of your Bank for the period of April 1, 2024 to March 31, 2025. It offers a holistic view of your Bank's operations, outlining the key activities that drive short- term and long-term value creation for its stakeholders. The Report also covers your Bank's product portfolio, competitive positioning, strategic priorities, business model, and risk management framework. <!-- PageNumber="02" --> <!-- PageBreak -->

Result 4
Ticker: BANDHAN
Company: Bandhan
Section: Bandhan > bandhan_annual_report.pdf > Page 180 > Please indicate material responsible business conduct and sustainability issues pertaining to environmental and social matters that present a risk or an opportunity to your business, rationale for identifying the same, approach to adapt or mitigate the risk, along with its financial implications, as per the following format:
RRF Score: 0.015625
Full-text Rank: None
Vector Distance: 0.5183801470586061
Content: relevant risks and opportunities that are mentioned below. This report includes information, which is material to all stakeholders of the Bank, and it presents an overview of the Bank's businesses and associated activities. The Bank discloses matters that substantially impact or affect the Bank's ability to create value. To ensure effective mitigation of identified risks, the Bank carries out a materiality analysis, to identify topics material to the Bank and its stakeholders. The sensitivity of an issue to stakeholders and the Bank, in terms of importance, forms the basis of the materiality analysis, which in turn guides the processes for identifying, managing, and devising specific action plans for addressing these material aspects. Every material topic is taken into account, and the Bank has policies and procedures in place to address these topics in order to create sustainable growth...

Result 5
Ticker: BANDHAN
Company: Bandhan
Section: Bandhan > bandhan_annual_report.pdf > Page 20
RRF Score: 0.015384615384615385
Full-text Rank: None
Vector Distance: 0.5255327728094081
Content: ### Page 20 <figure> <!-- PageHeader="Corporate Overview" --> </figure> <!-- PageHeader="Statutory Reports" --> <!-- PageHeader="Financial Statements" --> <figure> </figure> <figure> </figure> Reducing dependence on volatile bulk deposits has been a strategic focus, as these funds, while substantial, tend to be more expensive and less reliable. Your Bank's conscious shift towards a more stable deposit profile safeguards long-term cost efficiency and balance sheet stability. Your Bank also pivoted towards enhancing fee-based income, accelerating digital adoption, and reinforcing risk controls to sustain profitability and stakeholder confidence. Gross advances grew by 10% year-on-year to ₹1.37 lakh crore as of March 2025. This growth was broad-based, driven by increased demand from secured lending portfolios and strategic sectors aligned with national priorities. Secured advances showed ex...
```

## Create The RAG Agent

The agent is built through `market_analyst.services.agent`. Its retrieval tool calls the same shared hybrid-search function used above.

In [3]:
agent = build_market_analysis_agent(settings, retrieval_limit=RETRIEVAL_LIMIT)
print(type(agent))

<class 'langgraph.graph.state.CompiledStateGraph'>


## Ask A Fundamentals Question

The ticker is included in the prompt so the retrieval tool can stay scoped to the selected document.

In [6]:
user_prompt = f"""
Ticker: {selected_report.ticker}
Company: {selected_report.company_name}
Question: {QUESTION}

Answer using the RAG retrieval tool first. Cite the retrieved sections by section/page labels where available.
""".strip()

agent_result = agent.invoke({"messages": [{"role": "user", "content": user_prompt}]})
messages = agent_result["messages"]
final_message = messages[-1]
final_text = getattr(final_message, "content", str(final_message))

display(Markdown(final_text))

2026-05-10 12:35:29,379 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/gpt-5.4-mini/chat/completions?api-version=2025-04-01-preview "HTTP/1.1 200 OK"
2026-05-10 12:35:39,537 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2025-04-01-preview "HTTP/1.1 200 OK"
2026-05-10 12:35:44,578 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/gpt-5.4-mini/chat/completions?api-version=2025-04-01-preview "HTTP/1.1 200 OK"
2026-05-10 12:35:54,513 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2025-04-01-preview "HTTP/1.1 200 OK"
2026-05-10 12:35:54,816 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=

Using the retrieved annual-report chunks for **Bandhan (BANDHAN)**, here’s what they say about **growth, debt, cash flow, and key business risks**:

### Growth
The report describes **steady growth on both liabilities and assets**. In the “State of Affairs of the Bank” section, total deposits rose from **₹1,35,201.99 crore to ₹1,51,212.50 crore** in FY25, a **11.8% increase**, while net advances grew from **₹1,21,136.78 crore to ₹1,31,987.32 crore**, a **9.0% increase**. Total business increased **10.9%** to **₹2,88,207 crore**.  
- **Source:** *Bandhan > bandhan_annual_report.pdf > Page 80 > State of Affairs of the Bank*

The performance commentary also says deposits reached **₹1.51 lakh crore**, up about **12% YoY**, and gross advances grew **10% YoY to ₹1.37 lakh crore**. It attributes growth to a more stable deposit profile, fee income, digital adoption, and risk controls.  
- **Source:** *Bandhan > bandhan_annual_report.pdf > Page 20 > Performance*

### Debt / Borrowings
The borrowings schedule shows total borrowings fell to **₹11,13,84,927 thousand** from **₹16,37,15,240 thousand** year-on-year. Borrowings were mainly from **other banks**, **other institutions & agencies**, and **outside India**; secured borrowings were **₹7,50,000 thousand** in both years.  
- **Source:** *Bandhan > bandhan_annual_report.pdf > Page 230 > Schedule 4 - Borrowings*

The report also notes debt instruments such as **Non-Convertible Debentures** and **Certificates of Deposit** with ratings reaffirmed/revised during the year.  
- **Source:** *Bandhan > bandhan_annual_report.pdf > Page 100*

### Cash Flow
The cash flow statement is available on **Page 230**. The retrieved chunk shows **profit before taxation of ₹3,62,32,770 thousand** for FY25, up from **₹2,94,29,123 thousand** in FY24, along with provisions and contingencies of **₹3,76,54,115 thousand**.  
- **Source:** *Bandhan > bandhan_annual_report.pdf > Page 230 > Cash Flow Statement for the year ended March 31, 2025*

However, the current retrieved chunk does **not include the full operating/investing/financing cash flow totals**, so I can’t reliably summarize the final net cash movement from the available evidence alone.

### Key Business Risks
The report says the bank is exposed to multiple risks and has a **comprehensive Enterprise-wide Integrated Risk Management Framework** covering:
- **Credit Risk**
- **Market Risk**
- **Liquidity Risk**
- **Operational Risk**
- other risks as well  
- **Source:** *Bandhan > bandhan_annual_report.pdf > Page 170 > G. Risk and Concerns*

It also says the bank’s risk approach is to identify and manage **major and emerging risks** through ongoing assessment, measurement, monitoring, and escalation, with regular review by the Board and senior management.  
- **Source:** *Bandhan > bandhan_annual_report.pdf > Page 110 > Major Risks*

Operational resilience is highlighted too: the bank has a **Business Continuity Risk Management framework** to recover critical activities and systems within defined timelines and protect critical infrastructure.  
- **Source:** *Bandhan > bandhan_annual_report.pdf > Page 110*

### Bottom line
- **Growth:** solid, with deposits and advances both rising in FY25.
- **Debt:** borrowings declined materially year-on-year.
- **Cash flow:** the retrieved chunk confirms profitability and provisions, but not the full cash flow totals.
- **Risks:** the main stated risks are credit, market, liquidity, and operational risk, managed through a formal enterprise-wide framework.

If you want, I can also turn this into a **bullish/bearish investment read-through** based strictly on the retrieved report sections.

## Agent Tool Trace

This compact trace confirms whether the model called the RAG retrieval tool before answering.

In [ ]:
for index, message in enumerate(messages, start=1):
    message_type = getattr(message, "type", type(message).__name__)
    tool_calls = getattr(message, "tool_calls", None)
    name = getattr(message, "name", None)
    print(f"{index}. {message_type}" + (f" | {name}" if name else ""))
    if tool_calls:
        print("   tool_calls:", tool_calls)
    content = getattr(message, "content", "")
    if content and message_type != "ai":
        print("   content:", str(content)[:500].replace("\n", " "))

## Validation

These assertions keep the notebook honest without making it brittle.

In [ ]:
assert result.report_count == 1
assert result.chunk_count > 0
assert all(chunk.metadata.get("source_path") for chunk in result.chunks)
assert all(chunk.metadata.get("heading_path") for chunk in result.chunks)
assert len(result.vector_ids) == result.chunk_count
assert result.reports_rows == result.chunk_count
assert hybrid_results, "Hybrid retrieval should return at least one chunk for the selected report."
assert messages, "Agent result should include messages."
assert str(final_text).strip(), "Agent final response should not be empty."

print("RAG agent end-to-end notebook validation passed.")